# Behavioral Biometrics for Zero-Trust Web Architectures
## Continuous Authentication via Cursor Dynamics

**Author:** Nityanand KG  
**Affiliation:** Department of Computer Science and Engineering, CHRIST (Deemed to be University), Bangalore  
**Target Publication:** Elsevier / IEEE Cybersecurity Benchmark  

---

### Abstract & Research Problem
Traditional web authentication models depend almost exclusively on perimeter Point-of-Entry (PoE) mechanisms (e.g., passwords and MFA). Once initial verification succeeds, systems issue session tokens (e.g., JSON Web Tokens) that implicitly trust the client until explicit expiry. This leaves enterprise cloud platforms catastrophically vulnerable to **session hijacking, token theft, and unauthorized physical workstation takeover**.

This research implements an end-to-end continuous, passive verification pipeline using **Behavioral Biometrics derived from human-computer cursor interaction dynamics** on the benchmark **Balabit Mouse Dynamics Challenge** dataset.

### Pipeline Structure
1. **Preprocessing Hygiene:** Coordinate outlier filtering (IQR method) and timestamp regularization.
2. **25-D Feature Extraction:** Kinematics, angular curvature, click cadence, and spatial geometry.
3. **Model Benchmark:** Evaluating Random Forest, SVM (RBF), GBDT, MLP, k-NN, and One-Class Anomaly Detectors.
4. **Biometric Evaluation:** ROC Curves, Equal Error Rate (EER), FAR/FRR analysis.
5. **Zero-Trust Simulation:** Real-time dynamic risk scoring ($R_t$) and automated JWT token revocation upon session hijacking.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import CANONICAL_FEATURE_ORDER, FIGURES_DIR, TABLES_DIR
from src.data.loader import load_engineered_dataset
from src.models.baselines import get_baseline_models
from src.models.mlp import build_mlp_pipeline
from src.evaluation.metrics import evaluate_predictions, calculate_eer
from src.evaluation.plots import plot_roc_curves, plot_feature_importances
from src.zero_trust.token_manager import ZeroTrustTokenManager, PolicyAction

print("Project imports loaded successfully!")

## 1. Dataset Loading & Feature Inspection
We load the engineered 25-feature behavioral biometric matrix.

In [ ]:
features_df, metadata_df = load_engineered_dataset(PROJECT_ROOT / "data" / "engineered_features.csv")
print(f"Total sessions: {len(features_df)}")
print(f"Feature dimensions: {features_df.shape[1]}")
print(f"Enrolled user cohorts: {metadata_df['user_id'].nunique()}")
print(f"Class Distribution: {metadata_df['is_illegal'].value_counts().to_dict()}")
features_df.head(5)

## 2. Statistical Analysis: Univariate Feature Distributions
Let us visualize the velocity, acceleration, click rate, and pause ratio distributions distinguishing legitimate vs. imposter interaction patterns.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
palette = {0: "#2b5c8f", 1: "#c62828"}

plot_cols = ["avg_velocity", "avg_acceleration", "click_rate", "pause_ratio"]
titles = ["Average Velocity (px/s)", "Average Acceleration (px/s²)", "Click Rate (clicks/s)", "Pause Event Ratio"]

for ax, col, title in zip(axes.flatten(), plot_cols, titles):
    sns.kdeplot(data=features_df, x=col, hue=metadata_df["is_illegal"], palette=palette, fill=True, common_norm=False, ax=ax)
    ax.set_title(title)
    ax.set_xlabel(col)
    ax.grid(True, linestyle=":", alpha=0.6)

fig.suptitle("Empirical Distributions: Legitimate Owner (Blue) vs. Imposter Sessions (Red)", fontsize=14)
fig.tight_layout()
plt.show()

## 3. Pre-computed Benchmark Results
Let us inspect the cross-validated benchmark performance table exported by `experiments/run_benchmark.py`.

In [ ]:
benchmark_csv = TABLES_DIR / "benchmark_results.csv"
if benchmark_csv.exists():
    res_df = pd.read_csv(benchmark_csv)
    display(res_df)
else:
    print("Benchmark CSV not found. Run 'python -m experiments.run_benchmark' first.")

## 4. Zero-Trust Continuous Session Hijacking Simulation
We demonstrate passive identity verification in a live session. At $t = 80$s, an imposter takes over the terminal. The Zero-Trust Risk Engine tracks $R(t)$ via an Exponential Moving Average (EMA) and triggers token revocation.

In [ ]:
from experiments.run_continuous_simulation import run_simulation

# Run simulation for user7
run_simulation(target_user="user7", hijack_at_sec=80.0, max_duration_sec=160.0)

## 5. Visualizing the Zero-Trust Timeline
Displaying the generated session hijacking timeline with threshold zones.

In [ ]:
from IPython.display import Image, display

timeline_fig = FIGURES_DIR / "session_hijacking_timeline.png"
if timeline_fig.exists():
    display(Image(filename=str(timeline_fig)))
else:
    print("Timeline figure not found.")